# ML Zoomcamp 2026 — Homework 3
## Machine Learning for Classification

**Dataset:** `course_lead_scoring_2026.csv` (5000 rows)
**Target:** `converted` — whether a lead signed up to the platform.

This notebook solves all six questions of Homework 3 and documents the reasoning.

## 1. Setup and data preparation

- Missing categorical values are filled with `'NA'`.
- Missing numerical values are filled with `0.0`.

In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

df = pd.read_csv("data/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
TARGET = "converted"

features = df.drop(columns=[TARGET]).copy()
categorical = features.select_dtypes(include=["object"]).columns.tolist()
numerical = features.select_dtypes(exclude=["object"]).columns.tolist()

features[categorical] = features[categorical].fillna("NA")
features[numerical] = features[numerical].fillna(0.0)

print("categorical:", categorical)
print("numerical:  ", numerical)

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical:   ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


/tmp/ipykernel_673133/26417444.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = features.select_dtypes(include=["object"]).columns.tolist()


## Question 1 — Mode of `industry`

In [3]:
df["industry"].mode().iloc[0]

'technology'

**Answer:** `technology`

## Question 2 — Biggest correlation

Correlation matrix for the numerical features; only the suggested pairs are compared.

In [4]:
corr = features[numerical].corr().abs()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [5]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for a, b in pairs:
    print(f"{a} & {b}: {corr.loc[a, b]:.4f}")

interaction_count & lead_score: 0.9157
number_of_courses_viewed & lead_score: 0.7572
number_of_courses_viewed & interaction_count: 0.7216
annual_income & interaction_count: 0.1228


**Answer:** `interaction_count` and `lead_score` (0.916)

## Split the data

In [6]:
full_train, test = train_test_split(
    features.assign(**{TARGET: df[TARGET]}), test_size=0.2, random_state=42
)
train, val = train_test_split(full_train, test_size=0.25, random_state=42)

train_y = train.pop(TARGET)
val_y = val.pop(TARGET)
train.shape, val.shape, test.shape

((3000, 8), (1000, 8), (1000, 9))

## Question 3 — Mutual information (training set, rounded to 2 decimals)

In [7]:
mi = {col: round(mutual_info_score(train_y, train[col]), 2) for col in categorical}
mi

{'lead_source': 0.03,
 'industry': 0.0,
 'employment_status': 0.02,
 'location': 0.0}

In [8]:
max(mi, key=mi.get)

'lead_source'

**Answer:** `lead_source` (0.03)

## Question 4 — Logistic regression accuracy

One-hot encode categorical features, fit on train, evaluate on validation.

In [9]:
def train_model(train_x, train_y, val_x, c=1.0):
    dv = DictVectorizer(sparse=False)
    train_matrix = dv.fit_transform(train_x.to_dict(orient="records"))
    val_matrix = dv.transform(val_x.to_dict(orient="records"))
    model = LogisticRegression(
        solver="liblinear", C=c, max_iter=1000, random_state=42
    )
    model.fit(train_matrix, train_y)
    return model, val_matrix


model, val_matrix = train_model(train, train_y, val)
original_accuracy = accuracy_score(val_y, model.predict(val_matrix))
round(original_accuracy, 2)

0.65

**Answer:** `0.65`

## Question 5 — Feature elimination (least useful feature)

In [10]:
differences = {}
for col in ["lead_source", "number_of_courses_viewed", "interaction_count"]:
    m, x = train_model(
        train.drop(columns=[col]), train_y, val.drop(columns=[col])
    )
    acc = accuracy_score(val_y, m.predict(x))
    differences[col] = abs(original_accuracy - acc)

differences

{'lead_source': 0.0030000000000000027,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004}

In [11]:
min(differences, key=differences.get)

'number_of_courses_viewed'

**Answer:** `number_of_courses_viewed` (difference 0.002)

## Question 6 — Regularized logistic regression (best `C`)

In [12]:
scores = {}
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    m, x = train_model(train, train_y, val, c=c)
    scores[c] = round(accuracy_score(val_y, m.predict(x)), 3)

scores

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}

In [13]:
max(scores, key=lambda c: (scores[c], -c))

0.001

**Answer:** `0.001` (validation accuracy 0.645)

---
## Summary

| # | Question | Answer |
|---|----------|--------|
| 1 | Mode of `industry` | `technology` |
| 2 | Biggest correlation | `interaction_count` & `lead_score` |
| 3 | Biggest mutual information | `lead_source` |
| 4 | Logistic regression accuracy | `0.65` |
| 5 | Least useful feature | `number_of_courses_viewed` |
| 6 | Best `C` | `0.001` |